In [ ]:
from pathlib import Path
import pandas as pd
BASE_DIR = Path(__file__).resolve().parent
csv_races_path = BASE_DIR.parent  / "raw" / "races.csv"
races = pd.read_csv(csv_races_path , na_values=r"\N")
print(races.shape)
print(list(races.columns))

#valeurs manquantes
print(races.dtypes)
print(races.isna().sum())
print(races.isna().mean().round(3))
print(races.loc[races["fp1_date"].notna(), "year"].value_counts().sort_index())

# A1. time : première année renseignée, puis part renseignée par saison
print(races.loc[races["time"].notna(), "year"].min())
print(races["time"].notna().groupby(races["year"]).mean().round(2).loc[2014:])
# A2. fp1_date présente mais fp1_time absente : quelles années ?
print(races.loc[races["fp1_date"].notna() & races["fp1_time"].isna(), "year"].value_counts().sort_index())

# A3. Les week-ends sprint
print(races.loc[races["sprint_date"].notna(), ["year", "round", "date"]].sort_values(["year", "round"]))
m = races["fp1_date"].notna() & races["fp3_date"].isna()
print(m.sum(), (m & races["sprint_date"].notna()).sum())

#où commence l'enregistrement des fp
print(races.loc[races["time"].isna(), "year"].max())   # attendu : 2004
print((races["year"] < 2005).sum())                     # attendu : 731
cols = [c for c in races.columns if c.startswith(("fp", "quali", "sprint"))]
print({c: int(races.loc[races[c].notna(), "year"].min()) for c in cols})

#Combien de sprint? 
sprint = races["sprint_date"].notna()
print(sprint.sum())                                  # attendu : 30
print(sprint.groupby(races["year"]).sum().loc[2021:])  # attendu : 3, 3, 6, 6, 6, 6
print(sprint[races["year"] < 2021].sum())            # attendu : 0


In [ ]:

print(races[races["raceId"] == 1184].T)

In [ ]:
from pathlib import Path
import pandas as pd 

BASE_DIR = Path(__file__).resolve().parent
csv_circuits_path = BASE_DIR.parent  / "raw" / "circuits.csv"
circuits = pd.read_csv(csv_circuits_path , na_values=r"\N")
print(circuits.shape)
print(list(circuits.columns))
print(circuits.dtypes)
print(circuits.isna().sum())
circuits.head()

# Toutes les altitudes nulles ou négatives
print(circuits.loc[circuits["alt"] <= 0, ["circuitId", "name", "location", "country", "alt"]])

# Pour comparer : les circuits les plus bas
print(circuits.sort_values("alt")[["name", "country", "alt"]].head(10))

# Miami a-t-il des courses, et quand ?
miami_id = circuits.loc[circuits["name"].str.contains("Miami", case=False), "circuitId"]
print(races.loc[races["circuitId"].isin(miami_id), ["year", "round", "date"]])



In [ ]:
#Doublons races
print(races["raceId"].duplicated().sum())
print(races.duplicated().sum())
print(races.duplicated(subset=["year", "round"]).sum())
print(races.duplicated(subset=["date", "circuitId"]).sum())

In [ ]:
#Doublons circuits
print(circuits["circuitId"].duplicated().sum())
print(circuits["circuitRef"].duplicated().sum())
print(circuits["name"].duplicated().sum())
print(circuits.duplicated().sum())

In [ ]:

#distributions, textes, ordre des courses

# Années et manches
print(races[["year", "round"]].describe())
print(races.groupby("year")["round"].agg(["count", "max"]).query("count != max"))  # manches non consécutives

# Dates (plante si un format est anormal, c'est voulu)
d = pd.to_datetime(races["date"], format="%Y-%m-%d")
print(d.min(), d.max())
print((d.dt.year != races["year"]).sum())            # date incohérente avec year

# Ordre des courses
s = races.assign(d=d).sort_values(["year", "round"])
print(s["d"].is_monotonic_increasing)                # la date croît avec (year, round) ?
print((s["raceId"].diff() < 0).sum())                # nombre de cas où race_id recule qui veut dire ne pas trier en fct de id 

# Textes
print((races["name"] != races["name"].str.strip()).sum())
print(races["name"].str.contains(r"&\w+;").sum())
print(races["name"].nunique())
print(races["name"].value_counts().head(30))

print((races["name"] != races["name"].str.strip()).sum())
print(races["name"].nunique())
print((races["name"] != races["name"].str.strip()).sum())
print(races["name"].str.contains(r"&\w+;").sum())
print(races["name"].str.contains("  ").sum())                        # doubles espaces

rare = races["name"].value_counts().iloc[30:]                        # les 86 courses restantes
print(rare)
print(races.loc[races["name"].str.contains(r"[^A-Za-z0-9 ]"), "name"].unique())   # caractères non standard

print(races.loc[races["name"] == "Bahrain Grand Prix in Malaysia", ["race_id" if False else "raceId", "year", "round", "date", "circuitId"]])

print(races["time"].dropna().str.match(r"^\d{2}:\d{2}:\d{2}$").mean())   # attendu : 1.0
print(races["time"].value_counts().head(10))


print(races.loc[(races["circuitId"] == 2) & (races["raceId"] != 1184), "year"].max())
results = pd.read_csv("../data/raw/results.csv", na_values=r"\N")
print(1184 in set(results["raceId"]))

In [ ]:
# 1. Distributions numériques
print(circuits[["lat", "lng", "alt"]].describe())

# 2. Extrêmes d'altitude
print(circuits.sort_values("alt")[["name", "country", "alt"]].iloc[[0, 1, 2, -3, -2, -1]])

# 3. Altitudes nulles ou négatives (le point resté en attente)
print((circuits["alt"] <= 0).sum())
print(circuits.loc[circuits["alt"] <= 0, ["name", "country", "alt"]])

# 4. Coordonnées impossibles
print(((circuits["lat"].abs() > 90) | (circuits["lng"].abs() > 180)).sum())   # doit valoir 0

# 5. Textes : espaces en trop, entités HTML, doubles espaces
for col in ["circuitRef", "name", "location", "country"]:
    print(
        col,
        "| espaces :", (circuits[col] != circuits[col].str.strip()).sum(),
        "| entités HTML :", circuits[col].str.contains(r"&\w+;").sum(),
        "| doubles esp. :", circuits[col].str.contains("  ").sum(),
    )

# 6. Format de circuitRef (minuscules, chiffres, underscores)
print(circuits["circuitRef"].str.match(r"^[a-z0-9_]+$").all())

# 7. Caractères non standard (accents, tirets : légitimes ou parasites ?)
for col in ["name", "location", "country"]:
    print(col, circuits.loc[circuits[col].str.contains(r"[^A-Za-z0-9 ]"), col].unique())

# 8. Variantes d'écriture des pays
print(circuits["country"].value_counts())

print(circuits.loc[circuits["name"].str.contains("Nelson Piquet"), ["circuitId", "circuitRef", "location", "country", "lat", "lng", "alt"]])

print(circuits.loc[circuits["country"] == "United States", ["circuitId", "name", "location", "country"]])
print(circuits.loc[~circuits["circuitRef"].str.match(r"^[a-z0-9_]+$"), ["circuitId", "circuitRef", "name"]])

print(circuits.loc[circuits["country"] == "Brazil", ["circuitId", "circuitRef", "location", "lat", "lng", "alt"]])
print(races.loc[races["circuitId"] == 36, "year"].agg(["min", "max", "count"]))

In [ ]:
print("races sans circuit connu (doit valoir 0) :", (~races["circuitId"].isin(circuits["circuitId"])).sum())
print("circuits sans aucune course (info)       :", (~circuits["circuitId"].isin(races["circuitId"])).sum())
print("circuits utilisés en 2014+ (info)        :", races.loc[races["year"] >= 2014, "circuitId"].nunique())

print("résultats sans course connue (doit valoir 0) :", (~results["raceId"].isin(races["raceId"])).sum())
has = races["raceId"].isin(results["raceId"])
print("courses avec / sans résultat (contrat : 1164 / 8) :", has.sum(), (~has).sum())
print("dernière course avec résultats :", races.loc[has, "date"].max())
print(races.loc[~has, ["raceId", "year", "round", "date"]].sort_values(["year", "round"]))
